# SumBasic

## Introduction
This notebook demonstrates **SumBasic** for text summarization. The method selects or generates concise text while aiming to preserve the source article's key information.

### Objectives
- Load the shared article from `article.txt`.
- Generate a summary with this notebook's method.
- Evaluate the result against `reference_summary.txt` with shared ROUGE and BLEU metrics.

### Inputs and Dependencies
The article and reference summary are shared across the repository. Classical methods use the dependencies listed in `requirements.txt`; pretrained inference also downloads its model checkpoint.


In [1]:
import os, sys
from pathlib import Path
CURRENT_DIR = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
     if (candidate / "summarization_utils.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the repository root containing summarization_utils.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HOME", str(ROOT / ".model_cache"))
os.environ.setdefault("NLTK_DATA", str(ROOT / ".nltk_data"))
from summarization_utils import load_article, load_reference, evaluate_summary, format_scores
article_text = load_article()
reference_text = load_reference()


# SumBasic
**SumBasic** is a simple yet effective algorithm for text summarization that is based on a probabilistic model of sentence selection. Here are some advantages and disadvantages of the SumBasic algorithm for text summarization:

### Pros:
*	Simplicity: SumBasic is easy to understand and implement, requiring only basic probabilistic modeling and word frequency analysis.
*	Language independence: SumBasic is language-independent and can be applied to texts in any language.
*	Good for extractive summarization: SumBasic is well-suited for extractive summarization, where the summary consists of selected sentences from the original text.
*	Good for single-document summarization: SumBasic is effective at summarizing single documents, and can produce summaries that are accurate and relevant.

### Cons:
*	Limited coverage: SumBasic tends to focus on the most frequent words and sentences, and may miss important details that are less frequent.
*	Lack of coherence: SumBasic may produce summaries that lack coherence, especially when summarizing longer texts.
*	Inability to handle new information: SumBasic does not handle new information that is not present in the original text very well, which can lead to inaccuracies in the summary.
*	Limited customization: SumBasic is a simple algorithm with limited customization options, which may limit its flexibility in certain applications.

Overall, SumBasic is a useful algorithm for extractive summarization of single documents, and is easy to implement and understand. However, it may have limitations in terms of coverage, coherence, and handling new information, and may not be as effective for more complex summarization tasks. Proper tuning and feature selection can help mitigate some of its limitations.

These are the scores we achieved:

    ROUGE Score:
    Precision: 1.000
    Recall: 0.417
    F1-Score: 0.589

    BLEU Score: 0.621

## References

Here are some research papers related to the SumBasic algorithm for text summarization:

1. "Automatic text summarization by sentence extraction" by H. P. Luhn, in IBM Journal of Research and Development (1958)

1. "Sumbasic: A simple yet effective approach to single-document summarization" by A. Nenkova and K. McKeown, in Proceedings of the 2005 Human Language Technology Conference and Conference on Empirical Methods in Natural Language Processing (HLT-EMNLP)

1. "Sumbasic++: An efficient multi-document summarization approach with topic modeling" by D. Shang, J. Liu, and X. Li, in Proceedings of the 2018 Conference on Empirical Methods in Natural Language Processing (EMNLP)

These papers discuss various aspects of the SumBasic algorithm, including its effectiveness in producing high-quality summaries, its comparison with other techniques like LexRank and TextRank, and its extension to multi-document summarization using topic modeling.

The SumBasic algorithm is a simple and effective approach to extractive summarization that assigns weights to each sentence in the document based on its frequency in the text. The algorithm iteratively updates the sentence weights and selects the most important sentences for the summary.

The papers suggest that SumBasic is a powerful and computationally efficient approach to automatic text summarization, particularly for single-document summarization tasks. The algorithm's simplicity and intuitive nature make it easy to implement and adapt to different domains and languages.


In [2]:
# Optional original setup command (install through requirements.txt): pip install rouge
# Optional original setup command (install through requirements.txt): pip install nltk
from rouge import Rouge 
import nltk
import nltk.translate.bleu_score as bleu
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tuhin\Downloads\Text-Summarization-Models-
[nltk_data]     main\.nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[

True

In [3]:
def get_word_frequencies(text):
    """
    Calculates the frequency of each word in the text
    """
    stop_words = set(stopwords.words('english'))
    words = [word.lower() for word in word_tokenize(text) if word.isalpha() and word.lower() not in stop_words]
    freq = nltk.FreqDist(words)
    return freq

In [4]:
def get_sentence_scores(text, freq):
    """
    Calculates the score of each sentence in the text
    """
    sentences = sent_tokenize(text)
    scores = []
    for sentence in sentences:
        sentence_score = 0
        sentence_words = [word.lower() for word in word_tokenize(sentence) if word.isalpha()]
        for word in sentence_words:
            sentence_score += freq[word]
        sentence_score /= len(sentence_words)
        scores.append((sentence, sentence_score))
    return scores

In [5]:
def summarize(text, length):
    """
    Summarizes the text to the specified length using the SumBasic algorithm
    """
    freq = get_word_frequencies(text)
    summary = []
    while len(summary) < length:
        sentence_scores = get_sentence_scores(text, freq)
        top_sentence = max(sentence_scores, key=lambda x: x[1])[0]
        summary.append(top_sentence)
        # update frequency distribution by reducing frequency of words in the selected sentence
        for word in word_tokenize(top_sentence):
            if word.isalpha():
                freq[word.lower()] -= 1
    return ' '.join(summary)

In [6]:
text = load_article()

In [7]:
summary = summarize(text, 3)
print(summary)

Land temperatures ranked second highest and ocean temperatures third highest. The global surface average was 1.17 degrees Celsius above the twentieth-century average and 1.34 degrees above the 1850–1900 average. Five reached Category 5 strength.


In [8]:
rouge = Rouge()
scores = rouge.get_scores(summary, reference_text)
print("ROUGE Score:")
print("Precision: {:.3f}".format(scores[0]['rouge-1']['p']))
print("Recall: {:.3f}".format(scores[0]['rouge-1']['r']))
print("F1-Score: {:.3f}".format(scores[0]['rouge-1']['f']))

ROUGE Score:
Precision: 0.222
Recall: 0.098
F1-Score: 0.136


In [9]:
from summarization_utils import load_reference, evaluate_summary, format_scores
predicted_paragraph = summary
reference_summary = load_reference()
scores = evaluate_summary(str(predicted_paragraph), reference_summary)
score = scores["bleu4"]
print("Metrics:", format_scores(scores))

Metrics: rouge1_f1=0.206, rouge2_f1=0.019, rougeL_f1=0.150, bleu4=0.026


In [10]:
print("BLEU Score: {:.3f}".format(score))

BLEU Score: 0.026
